# step00_suna_inventory - SUNA Stage 0: frame inventory and integrity (P45_06)

First QA stage for the rosette SUNA nitrate, the SUNA analogue of the CTD inventory. It reads the raw
SUNA frames in `L0/SUNA/*.CSV`, documents what is in each file, and flags frames you should not trust
before anything downstream (merge, bottle bias) touches them.

**Why this first.** A SUNA does not measure nitrate. It measures a UV absorption spectrum (190 to 394 nm
here) and infers nitrate by fitting the 217 to 240 nm window to reference spectra for nitrate, bromide
and a baseline. So a nitrate value is only as good as its spectrum, its fit, and the instrument's health
at that moment. Stage 0 surfaces exactly those: spectrum quality, fit RMSE, lamp and temperature health,
and timing (Johnson and Coletti, 2002; Sakamoto, Johnson and Coletti, 2009; Johnson et al., 2018).

**Your frame format (SUNA V2, serial 2503), decoded from the files:**
- Each file is `SATFHR` header rows (instrument config) then `SATSLF` full-ASCII light frames.
- A frame is 286 comma fields: 11 lead fields, the 256-channel spectrum, then 19 housekeeping fields.
- Lead: frame+serial, date (YYYYDDD), time (decimal hours UTC), nitrate (uM), nitrate (mgN/L),
  absorbance 254, absorbance 350, bromide, spectrum average, **dark value**, integration factor.
- Housekeeping: internal / spectrometer / lamp temperature, cumulative lamp time, relative humidity,
  main / lamp / internal voltage, main current, four fit terms, **fit RMSE**, then the checksum.

**Two things found in your data, worth knowing now:**
- There are **no `SATSDF` dark frames** (the SUNA ran in Continuous mode). Dark correction therefore
  uses the per-frame `dark value` column, not separate dark spectra. That is normal for this mode.
- The onboard nitrate was computed with **`SNA2503E.cal`** (from the file headers), but your metadata
  folder holds **`SNA2503C.cal`**. That is fine for Stage 0, but for any later re-derivation from the
  spectra you want the matching cal (E). This notebook flags it if they differ.

Interactive (same spirit as the CTD explorers): a **spectrum explorer** (pick a file and a frame, see
its full spectrum with the 217 to 240 nm fit window shaded and its nitrate and RMSE), and a **health
timeline** per file.

Sources: Sea-Bird Scientific SUNA V2 manual (frame format, lamp behaviour); Johnson and Coletti (2002);
Sakamoto, Johnson and Coletti (2009); Johnson et al. (2018), Processing BGC-Argo nitrate at the DAC level.

In [ ]:
import re, sys
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------------ backend
INTERACTIVE_ZOOM = True
import matplotlib
try:
    get_ipython().run_line_magic("matplotlib", "widget" if INTERACTIVE_ZOOM else "inline")
except Exception:
    INTERACTIVE_ZOOM = False
    try: get_ipython().run_line_magic("matplotlib", "inline")
    except Exception: matplotlib.use("Agg")
if INTERACTIVE_ZOOM:
    try: plt.ioff()
    except Exception: pass
try:
    from IPython.display import display, Image
except Exception:
    def display(obj): print(obj)
    Image = None

# ------------------------------------------------------------------ resolve repo root
try:
    _anchor = Path(__file__).resolve()
except NameError:
    _anchor = Path.cwd()
for _p in (_anchor, *_anchor.parents):
    if (_p / "project_paths.py").exists():
        sys.path.insert(0, str(_p)); break
from project_paths import project_root, find_suna_cal

# ------------------------------------------------------------------ config
CRUISE_ID = "P45_06"                       # <-- the only switch per cruise
CTD_ROOT  = project_root()
CRUISE_DIR = CTD_ROOT / "cruises" / CRUISE_ID
SUNA_L0   = CRUISE_DIR / "L0" / "SUNA"
CAL_FILE  = find_suna_cal(cruise_id=CRUISE_ID, filename="SNA2503C.cal")
CAST_MAP  = CRUISE_DIR / "metadata" / "suna_cast_map.csv"
OUT_DIR   = CRUISE_DIR / "_audit" / "suna"; OUT_DIR.mkdir(parents=True, exist_ok=True)

NSPEC = 256; SPEC_START = 11
FIT_LO, FIT_HI = 217.0, 240.0

SAT_COUNTS = 60000
LOW_COUNTS = 2000
RMSE_HI    = 1.0e-3
HUM_HI     = 10.0
WARMUP_S   = 60.0

EXPLORER_FIGSIZE = (13.0, 5.0)
print("cruise:", CRUISE_ID, "| SUNA L0:", SUNA_L0, "| out:", OUT_DIR)

In [ ]:
# ------------------------------------------------------------------ readers
def parse_cal(path):
    """Return (wavelengths[NSPEC], header_dict) from a SUNA .cal file."""
    wl, hdr = [], {}
    try:
        lines = Path(path).read_text(errors="replace").splitlines()
    except Exception as e:
        print("could not read cal file:", e); return np.arange(NSPEC, dtype=float), {}
    for ln in lines:
        p = ln.split(",")
        if p[0] == "E" and len(p) >= 2:
            try: wl.append(float(p[1]))
            except ValueError: pass
        elif p[0] == "H" and len(p) >= 2:
            kv = ln[2:].strip()
            hdr.setdefault("_H", []).append(kv)
    return np.array(wl, float), hdr

def _ts(datefield, timedec):
    """SUNA date YYYYDDD + decimal-hour time -> UTC datetime."""
    y = int(float(datefield)) // 1000; doy = int(float(datefield)) % 1000
    return datetime(y, 1, 1) + timedelta(days=doy - 1, hours=float(timedec))

def parse_suna_file(path):
    """Parse one SUNA .CSV. Returns (header_dict, frames_DataFrame, spectra[nframes, NSPEC])."""
    fhr, rows, spec = {}, [], []
    for ln in Path(path).read_text(errors="replace").splitlines():
        if ln.startswith("SATFHR"):
            p = ln.split(",", 2)
            if len(p) == 3: fhr[p[1].strip()] = p[2].strip()
        elif ln.startswith("SATSLF"):
            f = ln.split(",")
            if len(f) < SPEC_START + NSPEC + 6:
                continue
            try:
                sp = np.array(f[SPEC_START:SPEC_START + NSPEC], float)
            except ValueError:
                continue
            tr = f[SPEC_START + NSPEC:]
            def g(i):
                try: return float(tr[i])
                except (ValueError, IndexError): return np.nan
            try:
                rows.append(dict(
                    time_utc=_ts(f[1], f[2]),
                    nitrate_uM=float(f[3]), nitrate_mgNL=float(f[4]),
                    abs254=float(f[5]), abs350=float(f[6]), bromide=float(f[7]),
                    spec_avg=float(f[8]), dark=float(f[9]), int_factor=float(f[10]),
                    T_int=g(0), T_spec=g(1), T_lamp=g(2), lamp_s=g(3), humidity=g(4),
                    V_main=g(5), V_lamp=g(6), V_int=g(7), I_main=g(8), fit_rmse=g(13)))
                spec.append(sp)
            except ValueError:
                continue
    df = pd.DataFrame(rows)
    return fhr, df, (np.array(spec) if spec else np.empty((0, NSPEC)))

def flag_frames(df, spec):
    """Add Stage-0 integrity flag columns to df (per frame). Returns df."""
    if df.empty: return df
    t0 = df["time_utc"].iloc[0]
    secs = (df["time_utc"] - t0).dt.total_seconds()
    peak = spec.max(axis=1) if spec.size else np.full(len(df), np.nan)
    df = df.copy()
    df["peak_counts"] = peak
    df["f_warmup"]    = secs < WARMUP_S
    df["f_saturated"] = peak > SAT_COUNTS
    df["f_lowsignal"] = peak < LOW_COUNTS
    df["f_highrmse"]  = df["fit_rmse"] > RMSE_HI
    df["f_humid"]     = df["humidity"] > HUM_HI
    df["flagged"] = df[["f_warmup","f_saturated","f_lowsignal","f_highrmse","f_humid"]].any(axis=1)
    return df

def load_all():
    """Parse every SUNA .CSV in L0/SUNA. Returns (per_file_df, frames_by_file, spectra_by_file, cal_wl)."""
    cal_wl, _ = parse_cal(CAL_FILE)
    if cal_wl.size != NSPEC:
        print(f"NOTE: cal has {cal_wl.size} wavelengths, expected {NSPEC}; using channel index for x.")
        cal_wl = np.arange(NSPEC, dtype=float)
    files = sorted(SUNA_L0.glob("*.CSV")) + sorted(SUNA_L0.glob("*.csv"))
    files = sorted(set(files))
    frames, spectra, inv = {}, {}, []
    cast_map = {}
    if CAST_MAP.exists():
        try:
            m = pd.read_csv(CAST_MAP)
            cols = {c.lower(): c for c in m.columns}
            if "suna_file" in cols and "cast_id" in cols:
                for _, r in m.iterrows():
                    cast_map[str(r[cols["suna_file"]]).strip()] = str(r[cols["cast_id"]]).strip()
        except Exception as e:
            print("could not read cast map:", e)
    for p in files:
        fhr, df, sp = parse_suna_file(p)
        if df.empty:
            print(f"  {p.name}: no light frames parsed"); continue
        df = flag_frames(df, sp)
        frames[p.name] = df; spectra[p.name] = sp
        onboard_cal = fhr.get("Calibration      File", "")
        inv.append(dict(
            file=p.name, cast_id=cast_map.get(p.name, ""),
            n_frames=len(df),
            start_utc=df["time_utc"].iloc[0], end_utc=df["time_utc"].iloc[-1],
            duration_s=round((df["time_utc"].iloc[-1] - df["time_utc"].iloc[0]).total_seconds(), 1),
            median_dt_s=round(df["time_utc"].diff().dt.total_seconds().median(), 3),
            nitrate_uM_med=round(df["nitrate_uM"].median(), 2),
            rmse_med=df["fit_rmse"].median(),
            peak_med=int(np.nanmedian(df["peak_counts"])),
            n_warmup=int(df["f_warmup"].sum()), n_saturated=int(df["f_saturated"].sum()),
            n_lowsignal=int(df["f_lowsignal"].sum()), n_highrmse=int(df["f_highrmse"].sum()),
            n_humid=int(df["f_humid"].sum()), n_flagged=int(df["flagged"].sum()),
            onboard_cal=onboard_cal, firmware=fhr.get("Firmware      Version", ""),
            mode=fhr.get("Operating        Mode", "")))
    per_file = pd.DataFrame(inv)
    return per_file, frames, spectra, cal_wl

In [ ]:
# ------------------------------------------------------------------ run the inventory
per_file, FRAMES, SPECTRA, CAL_WL = load_all()
if per_file.empty:
    print("No SUNA files parsed; check SUNA_L0 =", SUNA_L0)
else:
    per_file.to_csv(OUT_DIR / f"{CRUISE_ID}__suna_inventory.csv", index=False, encoding="utf-8-sig")
    with pd.option_context("display.width", 200, "display.max_columns", 50):
        print(per_file[["file","cast_id","n_frames","start_utc","duration_s","median_dt_s",
                        "nitrate_uM_med","rmse_med","peak_med","n_flagged","onboard_cal"]].to_string(index=False))
    print(f"\n{len(per_file)} file(s), {int(per_file['n_frames'].sum())} light frames total "
          f"-> {OUT_DIR / (CRUISE_ID + '__suna_inventory.csv')}")
    # cal-version check (onboard vs metadata)
    onboard = set(c for c in per_file["onboard_cal"] if c)
    meta = CAL_FILE.name
    if onboard and not any(meta.split('.')[0][-1] == c.split('.')[0][-1] for c in onboard):
        print(f"\nNOTE: onboard nitrate used {sorted(onboard)} but metadata cal is {meta}. Fine for Stage 0; "
              f"for re-derivation from spectra later, use the matching cal.")
    # flag summary
    fl = per_file[["n_warmup","n_saturated","n_lowsignal","n_highrmse","n_humid"]].sum()
    print("\nflagged-frame totals:", ", ".join(f"{k.replace('n_','')}={int(v)}" for k, v in fl.items()))

In [ ]:
# ------------------------------------------------------------------ per-file health timeline (static PNGs)
def _new_fig(nr, nc, sharex=False, **kw):
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    fig = Figure(**kw); FigureCanvasAgg(fig)
    return fig, fig.subplots(nr, nc, squeeze=False, sharex=sharex)

def health_timeline_png(fname):
    df = FRAMES[fname]; t = (df["time_utc"] - df["time_utc"].iloc[0]).dt.total_seconds()
    fig, ax = _new_fig(4, 1, figsize=(11.5, 9.0), sharex=True)
    ax = ax.ravel()
    ax[0].plot(t, df["nitrate_uM"], lw=0.6, color="#2E6FB0"); ax[0].set_ylabel("NO3 (uM)")
    ax[0].axhline(0, color="#aaa", lw=0.6)
    ax[1].plot(t, df["fit_rmse"], lw=0.6, color="#B00"); ax[1].axhline(RMSE_HI, color="#f0a", lw=0.6, ls="--")
    ax[1].set_ylabel("fit RMSE")
    ax[2].plot(t, df["peak_counts"], lw=0.6, color="#4C9F70"); ax[2].axhline(SAT_COUNTS, color="#f0a", lw=0.6, ls="--")
    ax[2].set_ylabel("peak counts")
    ax[3].plot(t, df["T_int"], lw=0.6, label="internal"); ax[3].plot(t, df["T_lamp"], lw=0.6, label="lamp")
    ax[3].plot(t, df["humidity"], lw=0.6, color="#888", label="humidity %")
    ax[3].set_ylabel("T (C) / RH%"); ax[3].legend(fontsize=7, frameon=False, ncol=3)
    ax[3].set_xlabel("seconds from file start")
    fl = df["flagged"].to_numpy()
    for a in ax:
        for x in t[fl]:
            a.axvspan(x, x, color="#d62728", alpha=0.015)
        for s in ("top","right"): a.spines[s].set_visible(False)
    fig.suptitle(f"{CRUISE_ID}  SUNA health  -  {fname}  ({len(df)} frames, {int(df['flagged'].sum())} flagged)",
                 fontsize=11, y=0.995)
    fig.tight_layout(rect=[0,0,1,0.98])
    out = OUT_DIR / f"{CRUISE_ID}__suna_health__{Path(fname).stem}.png"
    fig.savefig(out, dpi=130); 
    try: import matplotlib.pyplot as _p; _p.close(fig)
    except Exception: pass
    return out

if not per_file.empty:
    for fname in FRAMES:
        png = health_timeline_png(fname)
        if Image is not None:
            try: display(Image(filename=str(png)))
            except Exception: pass
        print("  saved", png.name)

In [ ]:
# ------------------------------------------------------------------ interactive: spectrum explorer
# Pick a file and a frame; left = raw spectrum (lamp health, saturation, dark), right = absorbance
# RELATIVE to this file's own baseline, zoomed around the 217 to 240 nm fit window (this is where the
# nitrate signal lives, so frame-to-frame changes show up here even when the raw spectra look alike).
# All in memory, so stepping is fast. Uses interactive_output + a frame box + prev/next for reliable
# updates (the earlier slider-only build did not always redraw).
try:
    import ipywidgets as W
    _HAVE_W = True
except Exception:
    _HAVE_W = False
    print("ipywidgets not installed; pip install ipywidgets ipympl, then Restart Kernel + Run All.")

# per-file reference spectrum = median over UNFLAGGED frames; baseline frames dominate, so for a file
# with high-nitrate blocks this reference is effectively the low-nitrate spectrum. Relative, not an
# absolute calibrated absorbance (that needs a nitrate-free reference; that comes in Stage 1/2).
REF = {}
for _fn, _df in FRAMES.items():
    _ok = (~_df["flagged"]).to_numpy()
    REF[_fn] = np.median(SPECTRA[_fn][_ok], axis=0) if _ok.sum() >= 5 else np.median(SPECTRA[_fn], axis=0)

def _spectrum_png_bytes(fname, i):
    import io
    df = FRAMES[fname]; wl = CAL_WL
    i = int(max(0, min(i, len(df) - 1)))
    s = SPECTRA[fname][i]; r = df.iloc[i]; dk = r["dark"]
    num = np.clip(s - dk, 1, None); den = np.clip(REF[fname] - dk, 1, None)
    A = -np.log10(num / den)                     # relative absorbance vs file baseline
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    fig = Figure(figsize=EXPLORER_FIGSIZE); FigureCanvasAgg(fig)
    ax1, ax2 = fig.subplots(1, 2)
    ax1.axvspan(FIT_LO, FIT_HI, color="#2E6FB0", alpha=0.10)
    ax1.plot(wl, s, lw=0.9, color="#333")
    ax1.axhline(dk, color="#c060c0", lw=0.7, ls="--", label=f"dark {dk:.0f}")
    ax1.set_ylim(0, 66000); ax1.set_xlabel("wavelength (nm)"); ax1.set_ylabel("intensity (counts)")
    ax1.set_title("raw spectrum (fixed scale)", fontsize=9); ax1.legend(fontsize=7, frameon=False)
    m = (wl >= 205) & (wl <= 255)
    ax2.axvspan(FIT_LO, FIT_HI, color="#2E6FB0", alpha=0.10, label="fit window 217-240 nm")
    ax2.plot(wl[m], A[m], lw=1.2, color="#B00"); ax2.axhline(0, color="#aaa", lw=0.6)
    ax2.set_xlabel("wavelength (nm)"); ax2.set_ylabel("absorbance vs file baseline (relative)")
    ax2.set_title("absorbance near fit window", fontsize=9); ax2.legend(fontsize=7, frameon=False)
    for ax in (ax1, ax2):
        for sp_ in ("top", "right"): ax.spines[sp_].set_visible(False)
    flags = [k[2:] for k in ("f_warmup","f_saturated","f_lowsignal","f_highrmse","f_humid") if r.get(k)]
    ttl = (f"{fname}  frame {i}/{len(df)-1}  @ {r['time_utc'].strftime('%H:%M:%S')}   "
           f"NO3 {r['nitrate_uM']:.2f} uM   RMSE {r['fit_rmse']:.2e}   peak {r['peak_counts']:.0f}")
    if flags: ttl += "   FLAGS: " + ",".join(flags)
    fig.suptitle(ttl, fontsize=9.5, color=("#b00" if flags else "#222"))
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=120); buf.seek(0)
    try: import matplotlib.pyplot as _p; _p.close(fig)
    except Exception: pass
    return buf.getvalue()

if _HAVE_W and not per_file.empty:
    _files = list(FRAMES); _n0 = len(FRAMES[_files[0]]) - 1
    wfile  = W.Dropdown(options=_files, value=_files[0], description="file")
    wframe = W.IntSlider(min=0, max=_n0, value=0, description="frame",
                         continuous_update=False, layout=W.Layout(width="55%"))
    wnum   = W.BoundedIntText(min=0, max=_n0, value=0, description="go to",
                              layout=W.Layout(width="150px"))
    bprev  = W.Button(description="< prev", layout=W.Layout(width="80px"))
    bnext  = W.Button(description="next >", layout=W.Layout(width="80px"))
    W.link((wframe, "value"), (wnum, "value"))      # kernel-side link: fires observers reliably
    def _sync_file(*_):
        n = len(FRAMES[wfile.value]) - 1
        wframe.max = n; wnum.max = n
        if wframe.value > n: wframe.value = 0
    wfile.observe(_sync_file, "value")
    bprev.on_click(lambda _: setattr(wframe, "value", max(0, wframe.value - 1)))
    bnext.on_click(lambda _: setattr(wframe, "value", min(wframe.max, wframe.value + 1)))
    def _render(fname, i):
        display(Image(data=_spectrum_png_bytes(fname, i)))
    out = W.interactive_output(_render, {"fname": wfile, "i": wframe})
    display(W.VBox([W.HBox([wfile, bprev, bnext]), W.HBox([wframe, wnum]), out]))